# Solar anomaly detection — deployment handoff

This notebook rebuilds the production predictor from the phase-A tables, calibrates the alert threshold on clean validation rows, and writes the files needed by the deployment team.

In [1]:
import json
import os
import random
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FEATURES = [
    "irr_wm2", "temp_c", "mod_temp_c", "solar_elev_deg",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos",
]
TARGET = "plant_pu"
TARGET_MAIN = TARGET
SPLITS = ("train", "val", "test")
STEP = pd.Timedelta("15min")


## 1. Load the prepared tables

In [2]:
tables = {
    split: pd.read_csv(
        f"phase_a/table_{split}.csv.gz",
        index_col="time_utc",
        parse_dates=True,
    )
    for split in SPLITS
}
scaler = json.load(open("phase_a/scaler_clean.json"))

expected_rows = {"train": 210439, "val": 35040, "test": 35033}
assert {split: len(df) for split, df in tables.items()} == expected_rows
print({split: df.shape for split, df in tables.items()})


{'train': (210439, 13), 'val': (35040, 13), 'test': (35033, 13)}


## 2. Build model features

In [3]:
def add_time_features(df):
    out = df.copy()
    idx = pd.DatetimeIndex(out.index).tz_convert("UTC")
    hour = idx.hour.to_numpy() + idx.minute.to_numpy() / 60
    doy = idx.day_of_year.to_numpy()
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    return out

params = scaler["params"]
def scale_x(df):
    med = pd.Series({c: params[c]["median"] for c in FEATURES})
    scale = pd.Series({c: params[c]["scale"] for c in FEATURES})
    return ((df[FEATURES] - med) / scale).to_numpy(np.float32)

def unscale_y(values):
    return values * params[TARGET]["scale"] + params[TARGET]["median"]

for split in SPLITS:
    tables[split] = add_time_features(tables[split])


## 3. Select clean training rows

In [4]:
clean = {split: df[df["clean"] == 1].copy() for split, df in tables.items()}
expected_clean = {"train": 83135, "val": 12146, "test": 8857}
assert {split: len(df) for split, df in clean.items()} == expected_clean

X_train = scale_x(clean["train"])
y_train = ((clean["train"][TARGET] - params[TARGET]["median"]) / params[TARGET]["scale"]).to_numpy(np.float32)
X_val = scale_x(clean["val"])
y_val = ((clean["val"][TARGET] - params[TARGET]["median"]) / params[TARGET]["scale"]).to_numpy(np.float32)


## 4. Train the GBM

In [5]:
runs = {}
models = {}
for num_leaves in (15, 31, 63):
    candidate = lgb.LGBMRegressor(
        objective="regression",
        learning_rate=0.05,
        num_leaves=num_leaves,
        min_child_samples=100,
        colsample_bytree=1.0,
        n_estimators=2000,
        random_state=SEED,
        deterministic=True,
        force_row_wise=True,
        n_jobs=1,
        verbosity=-1,
    )
    candidate.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    runs[num_leaves] = float(candidate.best_score_["valid_0"]["l2"])
    models[num_leaves] = candidate

best_leaves = min(runs, key=runs.get)
model = models[best_leaves]
print({"best_num_leaves": best_leaves, "best_val_mse": runs[best_leaves]})
assert model.booster_.num_feature() == len(FEATURES)


/home/ubuntu/.local/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


/home/ubuntu/.local/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


/home/ubuntu/.local/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{'best_num_leaves': 31, 'best_val_mse': 0.004073002815316493}


## 5. Score clean validation rows

In [6]:
def predict_pu(df):
    return unscale_y(model.predict(scale_x(df)))

val_clean = clean["val"].copy()
val_clean["yhat_pu"] = predict_pu(val_clean)
val_clean["resid"] = val_clean[TARGET] - val_clean["yhat_pu"]
inner_edges = np.quantile(val_clean["solar_elev_deg"], np.arange(1, 8) / 8)
val_clean["slice"] = np.searchsorted(inner_edges, val_clean["solar_elev_deg"], side="right")
med_s = np.zeros(8)
sig_s = np.zeros(8)
for s in range(8):
    resid_s = val_clean.loc[val_clean["slice"] == s, "resid"].to_numpy()
    med_s[s] = np.median(resid_s)
    sig_s[s] = 1.4826 * np.median(np.abs(resid_s - med_s[s]))
    if sig_s[s] == 0:
        sig_s[s] = np.finfo(float).eps

val_clean["score_pt"] = np.abs((val_clean["resid"] - med_s[val_clean["slice"]]) / sig_s[val_clean["slice"]])
val_clean["score_sm4"] = val_clean["score_pt"].rolling(4, min_periods=1).mean()
thresholds = {"threshold_sm4": float(val_clean["score_sm4"].quantile(0.995))}
res = {"thr": thresholds}
print(pd.Series(thresholds))


def solar_elevation(t_utc, lat=40.5, lon=21.8):
    n = (pd.DatetimeIndex(t_utc) - pd.Timestamp("2000-01-01 12:00", tz="UTC")) / pd.Timedelta(days=1)
    n = np.asarray(n, dtype=float)
    L = np.deg2rad((280.460 + 0.9856474 * n) % 360)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360)
    lam = L + np.deg2rad(1.915) * np.sin(g) + np.deg2rad(0.020) * np.sin(2 * g)
    eps = np.deg2rad(23.439 - 0.0000004 * n)
    dec = np.arcsin(np.sin(eps) * np.sin(lam))
    ra = np.arctan2(np.cos(eps) * np.sin(lam), np.cos(lam))
    lst = np.deg2rad(((280.46061837 + 360.98564736629 * n) + lon) % 360)
    ha = lst - ra
    la = np.deg2rad(lat)
    return np.rad2deg(np.arcsin(np.sin(la) * np.sin(dec) + np.cos(la) * np.cos(dec) * np.cos(ha)))

def predict(df):
    required = {"irr_wm2", "temp_c", "mod_temp_c", "plant_pu"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("df must have a UTC DatetimeIndex")
    features = df[["irr_wm2", "temp_c", "mod_temp_c"]].copy()
    features["solar_elev_deg"] = solar_elevation(df.index)
    hour = df.index.hour.to_numpy() + df.index.minute.to_numpy() / 60
    doy = df.index.day_of_year.to_numpy()
    features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    features["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    features["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    out = df[["irr_wm2", "temp_c", "mod_temp_c", "plant_pu"]].copy()
    out["solar_elev_deg"] = features["solar_elev_deg"]
    out["yhat_pu"] = unscale_y(model.predict(scale_x(features)))
    out["resid"] = out["plant_pu"] - out["yhat_pu"]
    out["slice"] = np.searchsorted(inner_edges, out["solar_elev_deg"], side="right")
    out["score_pt"] = np.abs((out["resid"] - med_s[out["slice"]]) / sig_s[out["slice"]])
    out["score_sm4"] = out["score_pt"].rolling(4, min_periods=1).mean()
    out["alarm"] = out["score_sm4"] >= res["thr"]["threshold_sm4"]
    return out


threshold_sm4    4.616006
dtype: float64


## 6. Check the scoring output

In [7]:
def score_table(df):
    return predict(df)

val_scored = score_table(clean["val"][[*FEATURES, TARGET]])
print(val_scored[["yhat_pu", "score_pt", "score_sm4", "alarm"]].describe())


            yhat_pu      score_pt     score_sm4
count  12146.000000  12146.000000  12146.000000
mean       0.450101      1.034861      1.034693
std        0.230742      1.188530      0.894483
min        0.013016      0.000000      0.028405
25%        0.245529      0.306455      0.428658
50%        0.490118      0.665261      0.728138
75%        0.656530      1.342346      1.357075
max        0.849449     15.997878      9.820560


## 7. Deployment interface

### 7.1 Input contract

The deployment input is a UTC-indexed dataframe with `irr_wm2`, `temp_c`, `mod_temp_c`, and `plant_pu`. The target is retained because the residual score compares the observed plant output with the GBM estimate.

In [8]:
def solar_elevation(t_utc, lat=40.5, lon=21.8):
    n = (pd.DatetimeIndex(t_utc) - pd.Timestamp("2000-01-01 12:00", tz="UTC")) / pd.Timedelta(days=1)
    n = np.asarray(n, dtype=float)
    L = np.deg2rad((280.460 + 0.9856474 * n) % 360)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360)
    lam = L + np.deg2rad(1.915) * np.sin(g) + np.deg2rad(0.020) * np.sin(2 * g)
    eps = np.deg2rad(23.439 - 0.0000004 * n)
    dec = np.arcsin(np.sin(eps) * np.sin(lam))
    ra = np.arctan2(np.cos(eps) * np.sin(lam), np.cos(lam))
    lst = np.deg2rad(((280.46061837 + 360.98564736629 * n) + lon) % 360)
    ha = lst - ra
    la = np.deg2rad(lat)
    return np.rad2deg(np.arcsin(np.sin(la) * np.sin(dec) + np.cos(la) * np.cos(dec) * np.cos(ha)))


### 7.2 Prediction function

In [9]:
def predict(df):
    required = {"irr_wm2", "temp_c", "mod_temp_c", "plant_pu"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("df must have a UTC DatetimeIndex")

    features = df[["irr_wm2", "temp_c", "mod_temp_c"]].copy()
    features["solar_elev_deg"] = solar_elevation(df.index)
    hour = df.index.hour.to_numpy() + df.index.minute.to_numpy() / 60
    doy = df.index.day_of_year.to_numpy()
    features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    features["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    features["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

    out = df[["irr_wm2", "temp_c", "mod_temp_c", "plant_pu"]].copy()
    out["solar_elev_deg"] = features["solar_elev_deg"]
    out["yhat_pu"] = unscale_y(model.predict(scale_x(features)))
    out["resid"] = out["plant_pu"] - out["yhat_pu"]
    out["slice"] = np.searchsorted(inner_edges, out["solar_elev_deg"], side="right")
    out["score_pt"] = np.abs((out["resid"] - med_s[out["slice"]]) / sig_s[out["slice"]])
    out["score_sm4"] = out["score_pt"].rolling(4, min_periods=1).mean()
    out["alarm"] = out["score_sm4"] >= res["thr"]["threshold_sm4"]
    return out


### 7.3 Threshold used in production

In [10]:
production_threshold = res["thr"]["threshold_sm4"]
print(f"score_sm4 threshold: {production_threshold:.6f}")


score_sm4 threshold: 4.616006


## 8. Arrays appendix removed for deployment.

## Notes
- Test data was never used for calibration.

## Export Artifacts

In [11]:
artifact_dir = Path("model_artifacts")
artifact_dir.mkdir(exist_ok=True)
model.booster_.save_model("model_artifacts/model.txt")

with open(artifact_dir / "scaler.json", "w") as f:
    json.dump(scaler, f, indent=2)
with open(artifact_dir / "slices.json", "w") as f:
    json.dump({
        "inner_edges": inner_edges.tolist(),
        "med_s": med_s.tolist(),
        "sig_s": sig_s.tolist(),
        "features": FEATURES,
        "target": TARGET,
    }, f, indent=2)
with open(artifact_dir / "thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)


In [12]:
sample = clean["train"].sample(100, random_state=42)[["irr_wm2", "temp_c", "mod_temp_c", TARGET]].copy()
sample.index.name = "time_utc"
sample.to_csv("sample_input.csv")
expected = predict(sample)
expected.to_csv("expected_output.csv")
print({"sample_input.csv": sample.shape, "expected_output.csv": expected.shape})


{'sample_input.csv': (100, 4), 'expected_output.csv': (100, 11)}
